In [ ]:
!pip install -q gTTS
!pip install -q git+https://github.com/openai/whisper.git
!apt-get install -y ffmpeg


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.0 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:

import getpass

ELEVENLABS_API_KEY = getpass.getpass("ElevenLabs API Key: ")

ElevenLabs API Key: ··········


In [ ]:
import requests

def list_elevenlabs_voices():
    url = "https://api.elevenlabs.io/v1/voices"
    headers = {"xi-api-key": ELEVENLABS_API_KEY}

    response = requests.get(url, headers=headers)
    print("Status code:", response.status_code)

    if response.status_code != 200:
        print(response.text)
        return

    voices = response.json().get("voices", [])

    for voice in voices:
        name = voice.get("name", "")
        voice_id = voice.get("voice_id", "")

        if "Roger" in name:
            print("✅ الصوت المطلوب:")
            print("Voice name:", name)
            print("Voice ID:", voice_id)
            print("-" * 40)

list_elevenlabs_voices()

Status code: 200
✅ الصوت المطلوب:
Voice name: Roger - Laid-Back, Casual, Resonant
Voice ID: CwhRBWXzGAHq8TQ4Fs17
----------------------------------------


In [ ]:
from gtts import gTTS
import IPython.display as ipd
from google.colab import files
import json
import re
import os
import whisper

model = whisper.load_model("base")

print("✅ Whisper model loaded successfully")

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 160MiB/s]


✅ Whisper model loaded successfully


In [ ]:
import whisper
from gtts import gTTS
import IPython.display as ipd
from google.colab import files
import json
import re
import os

# تحميل نموذج Whisper
# 'base' will offer better accuracy than 'tiny'
model = whisper.load_model("base")

print("✅ Whisper model loaded successfully")

✅ Whisper model loaded successfully


In [ ]:
ELEVENLABS_VOICE_ID = "CwhRBWXzGAHq8TQ4Fs17"

In [ ]:

lexicon_data = {
    "رصين": {
        "word": "رصين",
        "root": "ر ص ن",
        "meanings": [
            {
                "meaning": "ثابت ومحكم وذو وقار",
                "example": "كان حديثه رصينًا وواضحًا.",
                "synonyms": ["متزن", "وقور", "محكم"],
                "antonyms": ["مضطرب", "ضعيف"],
                "source": "ملف تجريبي"
            }
        ]
    },
    "بصيرة": {
        "word": "بصيرة",
        "root": "ب ص ر",
        "meanings": [
            {
                "meaning": "إدراك عميق وفهم نافذ للأمور",
                "example": "يمتلك القائد بصيرة في اتخاذ القرار.",
                "synonyms": ["فطنة", "وعي", "إدراك"],
                "antonyms": ["غفلة", "جهل"],
                "source": "ملف تجريبي"
            }
        ]
    },
    "استنباط": {
        "word": "استنباط",
        "root": "ن ب ط",
        "meanings": [
            {
                "meaning": "استخراج معنى أو حكم من دليل أو قرينة",
                "example": "استنبط الباحث النتيجة من النص.",
                "synonyms": ["استخراج", "استنتاج"],
                "antonyms": ["نقل مباشر"],
                "source": "ملف تجريبي"
            }
        ]
    }
}

with open("lexicon.json", "w", encoding="utf-8") as f:
    json.dump(lexicon_data, f, ensure_ascii=False, indent=2)

print("✅ Sample lexicon.json created")


✅ Sample lexicon.json created


In [ ]:
import requests
import IPython.display as ipd

ELEVENLABS_VOICE_ID = "CwhRBWXzGAHq8TQ4Fs17"

def text_to_speech_arabic(text, output_path="faseeh_response.mp3"):
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{ELEVENLABS_VOICE_ID}"

    headers = {
        "xi-api-key": ELEVENLABS_API_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "text": text,
        "model_id": "eleven_multilingual_v2",
        "voice_settings": {
            "stability": 0.65,
            "similarity_boost": 0.85,
            "style": 0.15,
            "use_speaker_boost": True
        }
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code != 200:
        print("❌ ElevenLabs Error:")
        print(response.status_code)
        print(response.text)
        return None

    with open(output_path, "wb") as f:
        f.write(response.content)

    print("✅ تم توليد الصوت من ElevenLabs")
    return ipd.Audio(output_path, autoplay=True)

In [ ]:

def load_lexicon(json_path="lexicon.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

lexicon = load_lexicon()

print("✅ Lexicon loaded")
print("عدد الكلمات:", len(lexicon))


✅ Lexicon loaded
عدد الكلمات: 3


In [ ]:


def normalize_arabic(text):
    text = text.strip()
    text = re.sub(r"[؟?،,.!]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


In [ ]:

def extract_intent_and_word(user_text, lexicon):
    text = normalize_arabic(user_text)

    # تحديد نوع السؤال
    if any(key in text for key in ["معنى", "ما معنى", "اش معنى"]):
        intent = "meaning"
    elif any(key in text for key in ["جذر", "اصل", "أصل"]):
        intent = "root"
    elif any(key in text for key in ["مرادف", "مرادفات"]):
        intent = "synonyms"
    elif any(key in text for key in ["ضد", "عكس", "أضداد", "اضداد"]):
        intent = "antonyms"
    elif any(key in text for key in ["مثال", "جملة"]):
        intent = "example"
    else:
        intent = "meaning"

    # استخراج الكلمة من السؤال بناءً على الكلمات الموجودة في المعجم
    found_word = None
    for word in lexicon.keys():
        if word in text:
            found_word = word
            break

    return intent, found_word



In [ ]:

def build_response(intent, word, lexicon):
    if not word:
        return "عذرًا، لم أتمكن من تحديد الكلمة المطلوبة. يمكنك أن تسأل مثل: ما معنى رصين؟"

    if word not in lexicon:
        return f"عذرًا، لم أجد الكلمة {word} في المعجم الحالي."

    entry = lexicon[word]
    meanings = entry.get("meanings", [])

    if not meanings:
        return f"وجدت الكلمة {word}، لكن لا توجد معانٍ مسجلة لها حاليًا."

    first_meaning = meanings[0]

    if intent == "meaning":
        response = f"معنى كلمة {word}: {first_meaning.get('meaning', 'غير متوفر')}."

        if len(meanings) > 1:
            response += f" ولهذه الكلمة {len(meanings)} معانٍ مسجلة."

    elif intent == "root":
        response = f"جذر كلمة {word} هو: {entry.get('root', 'غير متوفر')}."

    elif intent == "synonyms":
        synonyms = first_meaning.get("synonyms", [])
        if synonyms:
            response = f"من مرادفات كلمة {word}: " + "، ".join(synonyms) + "."
        else:
            response = f"لا توجد مرادفات مسجلة لكلمة {word} حاليًا."

    elif intent == "antonyms":
        antonyms = first_meaning.get("antonyms", [])
        if antonyms:
            response = f"من أضداد كلمة {word}: " + "، ".join(antonyms) + "."
        else:
            response = f"لا توجد أضداد مسجلة لكلمة {word} حاليًا."

    elif intent == "example":
        response = f"مثال على كلمة {word}: {first_meaning.get('example', 'غير متوفر')}."

    else:
        response = f"معنى كلمة {word}: {first_meaning.get('meaning', 'غير متوفر')}."

    source = first_meaning.get("source")
    if source:
        response += f" المصدر: {source}."

    return response


In [ ]:
uploaded = files.upload()

audio_file = list(uploaded.keys())[0]
print("✅ Uploaded file:", audio_file)

Saving faseeh_voice_assistant.mp3 to faseeh_voice_assistant (1).mp3
✅ Uploaded file: faseeh_voice_assistant (1).mp3


In [ ]:
def faseeh_voice_assistant(audio_file_path):
    print("🎧 جاري تحويل الصوت إلى نص...")

    result = model.transcribe(audio_file_path, language="ar")
    user_text = result["text"].strip()

    print("🗣️ النص المستخرج:")
    print(user_text)

    intent, word = extract_intent_and_word(user_text, lexicon)

    print("\n🔎 نوع الطلب:", intent)
    print("📌 الكلمة:", word)

    response = build_response(intent, word, lexicon)

    print("\n🤖 رد فصيح:")
    print(response)

    return text_to_speech_arabic(response)


faseeh_voice_assistant(audio_file)

🎧 جاري تحويل الصوت إلى نص...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


🗣️ النص المستخرج:
ممعنا رسين مجد ركالمة استنباط عطني مرادف بصيرة

🔎 نوع الطلب: synonyms
📌 الكلمة: بصيرة

🤖 رد فصيح:
من مرادفات كلمة بصيرة: فطنة، وعي، إدراك. المصدر: ملف تجريبي.
✅ تم توليد الصوت من ElevenLabs


In [ ]:

WAKE_WORDS = ["فصيح", "يا فصيح", "فسيح"]  # فسيح احتياط لو Whisper أخطأ بالكتابة

def has_wake_word(user_text):
    normalized = normalize_arabic(user_text)
    return any(wake_word in normalized for wake_word in WAKE_WORDS)


def remove_wake_word(user_text):
    cleaned = normalize_arabic(user_text)

    for wake_word in WAKE_WORDS:
        cleaned = cleaned.replace(wake_word, "")

    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned




In [ ]:
def faseeh_voice_assistant(audio_file_path):
    print("🎧 جاري تحويل الصوت إلى نص...")

    result = model.transcribe(audio_file_path, language="ar")
    user_text = result["text"].strip()

    print("🗣️ النص المستخرج:")
    print(user_text)

    # 1. التحقق من كلمة التنبيه
    if not has_wake_word(user_text):
        response = "نادني أولًا بقولك: فصيح، ثم اسأل عن معنى الكلمة أو جذرها أو مرادفها."
        print("\n🤖 رد فصيح:")
        print(response)
        return text_to_speech_arabic(response)

    # 2. حذف كلمة فصيح من السؤال
    clean_question = remove_wake_word(user_text)

    print("\n🧹 السؤال بعد حذف كلمة التنبيه:")
    print(clean_question)

    # 3. فهم نوع السؤال والكلمة
    intent, word = extract_intent_and_word(clean_question, lexicon)

    print("\n🔎 نوع الطلب:", intent)
    print("📌 الكلمة:", word)

    # 4. بناء الرد من المعجم
    response = build_response(intent, word, lexicon)

    print("\n🤖 رد فصيح:")
    print(response)

    # 5. تحويل الرد إلى صوت ElevenLabs
    return text_to_speech_arabic(response)


### How to use the `faseeh_voice_assistant` function to transcribe an audio file

First, upload your audio file (e.g., in MP3 or WAV format). Then, the `faseeh_voice_assistant` function will handle the transcription and provide a response.

In [ ]:
from google.colab import files

# Upload an audio file
print("Please upload your audio file for transcription:")
uploaded = files.upload()

# Get the name of the uploaded file
if uploaded:
    audio_file_to_transcribe = list(uploaded.keys())[0]
    print(f"✅ Uploaded file: {audio_file_to_transcribe}")

    # Call the assistant function with the uploaded file
    faseeh_voice_assistant(audio_file_to_transcribe)
else:
    print("❌ No file was uploaded.")

Please upload your audio file for transcription:


Saving faseeh_voice_assistant.mp3 to faseeh_voice_assistant (2).mp3
✅ Uploaded file: faseeh_voice_assistant (2).mp3
🎧 جاري تحويل الصوت إلى نص...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


🗣️ النص المستخرج:
ممعنا رسين مجد ركالمة استنباط عطني مرادف بصيرة

🤖 رد فصيح:
نادني أولًا بقولك: فصيح، ثم اسأل عن معنى الكلمة أو جذرها أو مرادفها.
✅ تم توليد الصوت من ElevenLabs


In [ ]:
uploaded = files.upload()

audio_file = list(uploaded.keys())[0]
print("✅/content/ElevenLabs_2026-05-05T12_20_32_My voice morooj_ivc_sp95_s60_sb68_se0_b_m2.mp3", audio_file)

faseeh_voice_assistant(audio_file)

Saving faseeh_voice_assistant.mp3 to faseeh_voice_assistant (3).mp3
✅/content/ElevenLabs_2026-05-05T12_20_32_My voice morooj_ivc_sp95_s60_sb68_se0_b_m2.mp3 faseeh_voice_assistant (3).mp3
🎧 جاري تحويل الصوت إلى نص...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


🗣️ النص المستخرج:
ممعنا رسين مجد ركالمة استنباط عطني مرادف بصيرة

🤖 رد فصيح:
نادني أولًا بقولك: فصيح، ثم اسأل عن معنى الكلمة أو جذرها أو مرادفها.
✅ تم توليد الصوت من ElevenLabs


In [ ]:
def faseeh_conversation_session():
    print("🎙️ مرحبًا، أنا فصيح.")
    print("ارفعي ملفًا صوتيًا يبدأ بكلمة: فصيح")
    print("مثال: فصيح، ما معنى رصين؟")
    print("-" * 50)

    while True:
        print("\n📤 ارفعي ملف السؤال الصوتي:")
        uploaded = files.upload()

        if not uploaded:
            print("لم يتم رفع أي ملف.")
            break

        audio_file = list(uploaded.keys())[0]
        print("✅ Uploaded file:", audio_file)

        print("\n🎧 جاري تحويل الصوت إلى نص...")
        result = model.transcribe(audio_file, language="ar")
        user_text = result["text"].strip()

        print("🗣️ النص المستخرج:")
        print(user_text)

        if not has_wake_word(user_text):
            response = "نادني أولًا بقولك: فصيح، ثم اسأل عن معنى الكلمة أو جذرها أو مرادفها."
            print("\n🤖 رد فصيح:")
            print(response)
            display(text_to_speech_arabic(response))
        else:
            clean_question = remove_wake_word(user_text)

            print("\n🧹 السؤال بعد حذف كلمة التنبيه:")
            print(clean_question)

            intent, word = extract_intent_and_word(clean_question, lexicon)

            print("\n🔎 نوع الطلب:", intent)
            print("📌 الكلمة:", word)

            response = build_response(intent, word, lexicon)

            print("\n🤖 رد فصيح:")
            print(response)
            display(text_to_speech_arabic(response))

        print("\n" + "-" * 50)
        answer = input("هل تريدين سؤالًا آخر؟ اكتبي نعم أو لا: ").strip().lower()

        if answer not in ["نعم", "ايه", "إيه", "yes", "y"]:
            goodbye = "شكرًا لاستخدامك فصيح. إلى اللقاء."
            print("\n🤖", goodbye)
            display(text_to_speech_arabic(goodbye))
            break


In [ ]:
faseeh_conversation_session()


🎙️ مرحبًا، أنا فصيح.
ارفعي ملفًا صوتيًا يبدأ بكلمة: فصيح
مثال: فصيح، ما معنى رصين؟
--------------------------------------------------

📤 ارفعي ملف السؤال الصوتي:


Saving faseeh_voice_assistant.mp3 to faseeh_voice_assistant (5).mp3
✅ Uploaded file: faseeh_voice_assistant (5).mp3

🎧 جاري تحويل الصوت إلى نص...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


🗣️ النص المستخرج:
ممعنا رسين مجد ركالمة استنباط عطني مرادف بصيرة

🤖 رد فصيح:
نادني أولًا بقولك: فصيح، ثم اسأل عن معنى الكلمة أو جذرها أو مرادفها.
✅ تم توليد الصوت من ElevenLabs



--------------------------------------------------
هل تريدين سؤالًا آخر؟ اكتبي نعم أو لا: نعم

📤 ارفعي ملف السؤال الصوتي:


لم يتم رفع أي ملف.
